# GEFCom Connecticut exploratory analysis

This notebook reads GridEval’s canonical processed CT dataset and split index files. It is read-only and does not rebuild or write processed data, split indices, or benchmark results. The weather relationship plot is descriptive exploration; the current benchmark’s default feature policy excludes weather, so this plot does not represent weather-informed model results.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "data" / "processed" / "gefcom_ct_clean.parquet").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate GridEval/data/processed/gefcom_ct_clean.parquet from the notebook's working directory.")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"


In [ ]:
ct = pd.read_parquet(PROCESSED_DIR / "gefcom_ct_clean.parquet")
if not isinstance(ct.index, pd.DatetimeIndex):
    raise ValueError("Canonical GEFCom CT data must use a DatetimeIndex.")
if not ct.index.is_monotonic_increasing or not ct.index.is_unique:
    raise ValueError("Canonical GEFCom timestamps must be unique and chronologically ordered.")
ct.head()


In [ ]:
print(f"GEFCom CT shape: {ct.shape}")
print(f"Date range: {ct.index.min()} to {ct.index.max()}")
print("Missing values:")
print(ct.isna().sum())


In [ ]:
valid = ct[["drybulb", "load"]].dropna()
poly_coefs = np.polyfit(valid["drybulb"], valid["load"], deg=2)
t_range = np.linspace(valid["drybulb"].min(), valid["drybulb"].max(), 200)
plt.figure(figsize=(9, 5))
plt.scatter(valid["drybulb"], valid["load"], alpha=0.08, color="#0284c7", s=3, label="Hourly observations")
plt.plot(t_range, poly_coefs[0] * t_range**2 + poly_coefs[1] * t_range + poly_coefs[2], color="#dc2626", linewidth=2.5, label="Descriptive quadratic fit")
plt.title("GEFCom CT: Exploratory Temperature–Load Relationship", fontsize=12)
plt.xlabel("Drybulb Temperature")
plt.ylabel("Zone Load (MW)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
split_names = ["train", "val", "test"]
split_indexes = {}
for split_name in split_names:
    split_path = PROCESSED_DIR / f"gefcom_{split_name}_idx.parquet"
    split_frame = pd.read_parquet(split_path)
    missing_keys = set(['timestamp']) - set(split_frame.columns)
    if missing_keys:
        raise ValueError(f"{split_path.name} is missing identity columns: {sorted(missing_keys)}")
    if "occurrence" in split_frame.columns:
        split_index = pd.MultiIndex.from_frame(split_frame[['timestamp']], names=["timestamp", "occurrence"])
        split_times = split_index.get_level_values("timestamp")
    else:
        split_index = pd.DatetimeIndex(split_frame["timestamp"], name="timestamp")
        split_times = split_index
    if not split_index.is_unique or not split_index.is_monotonic_increasing:
        raise ValueError(f"{split_path.name} must contain unique, ordered identities")
    split_indexes[split_name] = split_index
    print(f"GEFCom {split_name.title()}: {len(split_index):,} observations ({split_times.min()} to {split_times.max()})")

assert not (set(split_indexes["train"]) & set(split_indexes["val"]))
assert not (set(split_indexes["train"]) & set(split_indexes["test"]))
assert not (set(split_indexes["val"]) & set(split_indexes["test"]))
print("Canonical split index files are ordered, unique, and pairwise disjoint.")


## Safe rerun note

This notebook reads canonical processed data for exploration and reads the saved split index files only to summarize their observation counts and date boundaries. It does not construct train/validation/test datasets or write processed data, split indices, or benchmark results. Split creation and model data preparation remain in GridEval’s pipeline.
